# 1002 · 00 · Preparar el entorno

Laboratorio 1002 · Cuarentena y recuperación.

Este notebook hace tres cosas, y solo hay que ejecutarlo una vez (**Run all**):

1. Crea el catálogo `lab1002` con sus schemas y el volumen `landing.datos`.
2. Copia los ficheros de la carpeta `datos/` de este repositorio al volumen.
3. Imprime el YAML del workflow **con las rutas de tu carpeta ya puestas**, listo para pegar en *Edit as YAML*.

El laboratorio usa el script de ingesta del 1001 (`lab-1001-primer-workflow/src/01_ingesta.py`): no hace falta haber hecho el 1001, pero sí que esté en el mismo repositorio.

In [ ]:
import os
import shutil

CATALOGO = "lab1002"

# La carpeta de este notebook es la del laboratorio dentro de tu Git folder.
RUTA_LAB = os.getcwd()
if not RUTA_LAB.startswith("/Workspace"):
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    RUTA_LAB = "/Workspace" + os.path.dirname(ctx.notebookPath().get())
RUTA_TEMA = os.path.dirname(RUTA_LAB)

print("Laboratorio:", RUTA_LAB)
print("Catálogo:   ", CATALOGO)

## 1 · Catálogo, schemas y volumen

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
for schema in ['landing', 'bronze', 'silver', 'cuarentena']:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.landing.datos")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOGO}"))

## 2 · Datos al volumen

In [ ]:
origen = f"{RUTA_LAB}/datos"
destino = f"/Volumes/{CATALOGO}/landing/datos"

for fichero in sorted(os.listdir(origen)):
    shutil.copyfile(f"{origen}/{fichero}", f"{destino}/{fichero}")
    print("copiado", fichero)

display(dbutils.fs.ls(destino))

## 3 · El YAML de tu workflow

Copia **todo** el texto que aparece debajo de esta celda: lo pegarás en *Edit as YAML* cuando la guía te lo pida.

In [ ]:
with open(f"{RUTA_LAB}/workflow.yml", encoding="utf-8") as f:
    plantilla = f.read()

# Fuera los comentarios de cabecera: al job solo le interesa la definicion
plantilla = plantilla[plantilla.index("resources:"):]
yaml_listo = plantilla.replace("__RUTA_LAB__", RUTA_LAB).replace("__RUTA_TEMA__", RUTA_TEMA)
print(yaml_listo)